# SageMaker AI MLflow — Medical Triage Agent with Strands Agents

In this lab, you will build an intelligent **Medical Triage Agent** using the [Strands Agents SDK](https://github.com/strands-agents/sdk-python) and enable automatic tracing with MLflow.

**What you will learn:**
- Build a medical triage agent using Strands Agents SDK with custom tools
- Enable MLflow auto-tracing for Strands agents to capture agent reasoning and tool calls
- Observe agent traces in the SageMaker AI MLflow UI

### Prerequisites
- A running SageMaker AI endpoint from the previous lab (03-1) (`workshop-qwen3-8b`)
- A SageMaker AI MLflow App configured in the fundamentals lab

### Scenario Overview

You are building a **Medical Triage Agent** that assists healthcare staff by analyzing patient symptoms and recommending appropriate treatment protocols. The agent uses two tools:

1. **`symptom_lookup`** — Given a patient ID, retrieves the patient's reported symptoms from the records database
2. **`treatment_lookup`** — Given a set of symptoms, retrieves the matching condition diagnosis and step-by-step treatment protocol

The agent receives a patient ID, looks up their symptoms, identifies the likely condition, and returns the recommended triage level and treatment steps — all traced automatically in MLflow.

<div style="padding: 15px; background-color: #fff3cd; border-left: 5px solid #ffc107; color: #856404;">
<strong>⚠️ Important:</strong> The cell below installs libraries and restarts the kernel. After the restart, continue with the next cell.
</div>

In [ ]:
!pip install -U sagemaker==2.253.1 datasets==4.4.1 mlflow==3.9.0 fsspec==2023.9.2 --quiet

# restart kernel
import IPython
IPython.Application.instance().kernel.do_shutdown(True) #automatically restarts kernel

## 1. Setup Environment

We retrieve the endpoint name and MLflow tracking URI stored from previous labs.

In [ ]:
import json
import boto3
import sagemaker

sess = sagemaker.Session()

# Retrieve variables stored from previous labs
%store -r

print(f"Endpoint: {endpoint_name}")
print(f"MLflow Tracking URI: {MLFLOW_TRACKING_URI}")

## 2. Configure SageMaker AI MLflow

We configure MLflow to use a separate experiment for agent traces. This keeps agent traces separate from the DataCapture pipeline traces (which go to `medical-triage-datacapture`).

In [ ]:
import mlflow

agent_experiment_name = "medical-triage-agent"

# Set MLflow SDK to your configured MLflow App
mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)
experiment = mlflow.set_experiment(agent_experiment_name)

print(f"MLflow version: {mlflow.__version__}")
print(f"Tracking URI: {MLFLOW_TRACKING_URI}")
print(f"Experiment: {agent_experiment_name}")

### Enable Strands Auto-Tracing

`mlflow.strands.autolog()` automatically captures traces for every Strands agent invocation — including tool calls, model reasoning, and final responses — and logs them to your MLflow experiment.

In [ ]:
mlflow.strands.autolog()

## 3. Install Strands Agents SDK and Dependencies

In [ ]:
!pip install strands-agents "strands-agents[sagemaker]" strands-agents-tools --upgrade --quiet
!pip install mypy-boto3-sagemaker-runtime --quiet

## 4. Load Patient Data and Treatment Protocols

We use synthetic patient records and treatment protocols to simulate a medical triage system. Each patient has a unique ID, reported symptoms, age, and severity level.

In [ ]:
from data.data import patient_records, patient_ids
from data.solution_book import treatment_protocols

print(f"Loaded {len(patient_records)} patient records")
print(f"Loaded {len(treatment_protocols)} treatment protocols")

Let's examine the patient records. Each record contains a patient ID, their symptoms, age, and an initial severity assessment.

In [ ]:
patient_records

The treatment protocols map symptom combinations to conditions, triage levels, and step-by-step treatment plans.

In [ ]:
# Show one example protocol
example_symptoms = "chest pain, shortness of breath, sweating"
protocol = treatment_protocols[example_symptoms]
print(f"Symptoms: {example_symptoms}")
print(f"Condition: {protocol['condition']}")
print(f"Triage Level: {protocol['triage_level']}")
print(f"Protocol:")
for step in protocol['protocol']:
    print(f"  {step}")

## 5. Define Agent Tools

In Strands Agents, tools are Python functions decorated with `@tool` that the agent can invoke during its reasoning loop. We define two tools for our medical triage workflow.

In [ ]:
from strands import Agent, tool

### The `symptom_lookup` Tool

This tool allows the agent to look up a patient's symptoms by their patient ID. It returns the symptoms, age, and initial severity assessment.

In [ ]:
@tool()
def symptom_lookup(patient_id: str) -> str:
    """Look up patient symptoms by patient ID.

    Args:
        patient_id: The patient identifier (e.g., PT-001)

    Returns:
        A string with the patient's symptoms, age, and severity level.
    """
    patient_id = patient_id.upper().strip()
    for patient in patient_records:
        if patient['id'] == patient_id:
            return (
                f"Patient {patient['id']}: "
                f"Symptoms: {patient['symptoms']}. "
                f"Age: {patient['age']}. "
                f"Initial severity: {patient['severity']}."
            )
    return f"No patient found with ID: {patient_id}. Valid IDs range from PT-001 to PT-015."

### The `treatment_lookup` Tool

After identifying the patient's symptoms, the agent uses this tool to retrieve the matching condition diagnosis and treatment protocol.

In [ ]:
@tool()
def treatment_lookup(symptoms: str) -> str:
    """Retrieve treatment protocol based on patient symptoms.

    Args:
        symptoms: The patient's symptoms as a comma-separated string

    Returns:
        A string with the condition, triage level, and treatment steps.
    """
    symptoms_clean = symptoms.lower().strip()
    for key, protocol in treatment_protocols.items():
        if key.lower() == symptoms_clean:
            steps = "\n".join(protocol['protocol'])
            return (
                f"Condition: {protocol['condition']}\n"
                f"Triage Level: {protocol['triage_level']}\n"
                f"Treatment Protocol:\n{steps}"
            )
    return f"No matching treatment protocol found for symptoms: {symptoms}. Please consult a physician."

## 6. Create the Medical Triage Agent

Now we create the Strands agent using `SageMakerAIModel` as the model provider. Qwen3-8B returns tool calls in the standard OpenAI-compatible format, so no custom parsing is needed.

> **Note:** We disable streaming and use `/no_think` in the system prompt to get direct responses without reasoning traces, which keeps the agent loop clean.

In [ ]:
from strands.models.sagemaker import SageMakerAIModel
import boto3

# Create the SageMaker AI Model object
sagemaker_model = SageMakerAIModel(
    endpoint_config={
        "endpoint_name": endpoint_name,
    },
    payload_config={
        "max_tokens": 1024 * 5,
        "temperature": 0.3,
        "stream": False,
    },
    boto_session=boto3.Session(region_name=sess.boto_region_name),
)

print("SageMakerAIModel created successfully!")

In [ ]:
from strands import Agent

SYSTEM_PROMPT = """You are an expert medical triage assistant. /no_think

You help healthcare staff assess patients and recommend treatment protocols.

You have access to two tools:
1. symptom_lookup: Use this to retrieve a patient's symptoms by their patient ID
2. treatment_lookup: Use this to find the treatment protocol for a given set of symptoms

When given a patient ID:
1. First use symptom_lookup to get the patient's symptoms
2. Then use treatment_lookup with the EXACT symptom string returned by symptom_lookup (the part after 'Symptoms: ' and before the period)
3. Present the findings clearly: patient info, condition, triage level, and treatment steps

Always use the tools in sequence. Do not guess symptoms or treatments."""

# Create the agent
agent = Agent(
    model=sagemaker_model,
    tools=[symptom_lookup, treatment_lookup],
    system_prompt=SYSTEM_PROMPT,
)

print("Medical Triage Agent created successfully!")

## 7. Test the Agent

Let's test our medical triage agent with sample patient queries. The agent should:
1. Use `symptom_lookup` to retrieve the patient's symptoms
2. Use `treatment_lookup` to find the matching treatment protocol
3. Present a clear triage assessment

Each invocation is automatically traced by MLflow.

In [ ]:
result = agent("Can you help me triage patient PT-001?")

Let's test with another patient to see a different triage scenario.

In [ ]:
result = agent("Please assess patient PT-003 and provide the treatment protocol.")

Now let's test with a low-severity case to see how the agent handles non-emergency situations.

In [ ]:
result = agent("What is the triage assessment for patient PT-007?")

## 8. View Agent Traces in MLflow

Now you can view the detailed agent traces in the SageMaker AI MLflow UI.

1. Open your SageMaker AI MLflow UI
2. In the MLflow UI, navigate to the **Experiments** section
3. Select the **medical-triage-agent** experiment
4. Click on the **Traces** tab to see all captured traces

Each trace shows:
- The full agent reasoning loop
- Tool invocations with input arguments and return values
- Model calls with prompts and responses
- Timing information for each step

> **Tip:** Click on individual traces to see the detailed span tree, which shows exactly how the agent reasoned through each step.

> **Note:** These agent traces are in the `medical-triage-agent` experiment, separate from the DataCapture pipeline traces in `medical-triage-datacapture`.

## 9. Store Variables for Next Lab

We store the agent experiment name so it can be retrieved in the online evaluation notebook.

In [ ]:
%store agent_experiment_name

print(f"Stored agent_experiment_name = {agent_experiment_name}")

## Next Steps

You have built and tested a Medical Triage Agent with automatic MLflow tracing. In the next lab, you will evaluate these agent traces using LLM-as-a-Judge scorers and add human feedback.

> **Important:** Do not delete the endpoint — it will be used in subsequent labs.